# Assignment 21 — LangChain Document Loaders & Text Splitters

**Goal:** Load text, CSV, PDF, directory and web documents, then compare several LangChain text splitters.

> Run the cells after installing `requirements.txt`. The outputs shown by Jupyter after execution should be your actual execution evidence.


In [ ]:
from pathlib import Path
BASE = Path.cwd()
DATA = BASE / "data"

from langchain_community.document_loaders import (
    TextLoader, CSVLoader, PyPDFLoader, DirectoryLoader, WebBaseLoader
)
from langchain_text_splitters import (
    CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
)

print("REAL LANGCHAIN MODE")
print("Data folder:", DATA)


## Task 1 — TextLoader

In [ ]:
loader = TextLoader(str(DATA / "sample.txt"), encoding="utf-8")
docs = loader.load()
print("Number of documents:", len(docs))
print("Preview:", docs[0].page_content[:250])
print("Metadata:", docs[0].metadata)

## Task 2 — CSVLoader

In [ ]:
csv_docs = CSVLoader(str(DATA / "sample.csv")).load()
print("CSV documents:", len(csv_docs))
for i, d in enumerate(csv_docs[:3], 1):
    print(f"Row {i}:", d.page_content)

## Task 3 — PyPDFLoader

In [ ]:
pdf_docs = PyPDFLoader(str(DATA / "sample.pdf")).load()
print("PDF pages:", len(pdf_docs))
print("PDF preview:", pdf_docs[0].page_content[:350])
print("Metadata:", pdf_docs[0].metadata)

## Task 4 — DirectoryLoader

In [ ]:
directory_docs = DirectoryLoader(
    str(DATA), glob="*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
).load()
print("TXT documents:", len(directory_docs))
print([d.metadata.get("source") for d in directory_docs])

## Task 5 — WebBaseLoader

In [ ]:
try:
    web_docs = WebBaseLoader("https://example.com").load()
    print("Web documents:", len(web_docs))
    print(web_docs[0].page_content[:500])
except Exception as e:
    print("Actual network/loader error:", type(e).__name__, e)

## Task 6 — Why Text Splitting?

In [ ]:
print("Large documents must be split because LLMs have finite token limits and context windows.")
print("Smaller chunks improve retrieval precision and reduce irrelevant context.")
print("Chunk size balances useful context against token/context constraints.")

## Task 7 — CharacterTextSplitter

In [ ]:
splitter = CharacterTextSplitter(separator="\n", chunk_size=180, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print("Number of chunks:", len(chunks))
for i, c in enumerate(chunks[:3], 1):
    print(f"Chunk {i}:", c.page_content[:180].replace("\n", " "))

## Task 8 — RecursiveCharacterTextSplitter

In [ ]:
recursive = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
recursive_chunks = recursive.split_documents(docs)
print("Number of chunks:", len(recursive_chunks))
for i, c in enumerate(recursive_chunks[:3], 1):
    print(f"Chunk {i}:", c.page_content[:180].replace("\n", " "))
print("\nCharacter splitting follows its separator; recursive splitting tries multiple separators.")

## Task 9 — MarkdownHeaderTextSplitter

In [ ]:
markdown = (DATA / "sample.md").read_text(encoding="utf-8")
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
)
structured = header_splitter.split_text(markdown)
print("Structured chunks:", len(structured))
for i, c in enumerate(structured, 1):
    print(i, c.metadata, c.page_content[:150].replace("\n", " "))

## Task 10 — Semantic Splitting

In [ ]:
print("Semantic splitting groups text using meaning/similarity instead of only character boundaries.")
try:
    from langchain_experimental.text_splitter import SemanticChunker
    print("SemanticChunker import succeeded. An embeddings API/key is needed for a live demo.")
except Exception as e:
    print("Optional semantic splitter not available:", type(e).__name__, e)

## Task 11 — Unified Function

In [ ]:
def load_and_split_documents(path_or_url):
    if str(path_or_url).startswith(("http://", "https://")):
        documents = WebBaseLoader(str(path_or_url)).load()
    else:
        path = Path(path_or_url)
        if path.is_dir():
            documents = DirectoryLoader(
                str(path), glob="*.txt", loader_cls=TextLoader,
                loader_kwargs={"encoding": "utf-8"}
            ).load()
        elif path.suffix.lower() == ".pdf":
            documents = PyPDFLoader(str(path)).load()
        elif path.suffix.lower() == ".csv":
            documents = CSVLoader(str(path)).load()
        else:
            documents = TextLoader(str(path), encoding="utf-8").load()
    return RecursiveCharacterTextSplitter(
        chunk_size=180, chunk_overlap=30
    ).split_documents(documents)

local_chunks = load_and_split_documents(DATA)
print("Local directory chunks:", len(local_chunks))
print(local_chunks[0].page_content[:180])

try:
    web_chunks = load_and_split_documents("https://example.com")
    print("Web URL chunks:", len(web_chunks))
except Exception as e:
    print("Actual web error:", type(e).__name__, e)

## Task 12 — Observations & Insights

In [ ]:
print("Best general-purpose splitter for prose: RecursiveCharacterTextSplitter.")
print("Best when Markdown structure matters: MarkdownHeaderTextSplitter.")
print("Chunk overlap is important because it preserves context across chunk boundaries.")
print("Chunk size should be chosen with the model's token/context limits in mind.")

## Final observation

After running all cells, update `MY_RUN_NOTES.txt` with the exact outputs you observed, including any network or dependency errors and how you fixed them.
